<a href="https://colab.research.google.com/github/ApexShadow28/GenAI/blob/main/Homework/HW1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install & import the needed libraries

!pip install -q transformers torch

!pip install triton torchao



In [ ]:
import torch
import torch.nn.functional as F
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import os
os.environ["TQDM_DISABLE"] = "1" # Disables progress bar widgets error caused by GPT


In [ ]:
# Load tokenizer & model

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()


In [ ]:
# Enter your own input text

text = input("Enter a sentence: ")


In [ ]:
# The tokenization step typically creates subword tokens, and not necessarily whole words

tokens = tokenizer.encode(text, return_tensors="pt")

print("Token IDs:", tokens.tolist()[0])
print("Tokens:")
for tid in tokens[0]:
    print(f"{tid.item():>6} → '{tokenizer.decode(tid)}'")


In [ ]:
# The transformer forward pass ensures that each token now contains contextual information from previous tokens.
# This is the most important step conceptually, because this is where the model goes from isolated words to understanding a sentence.

with torch.no_grad():
    # Tokenize the input text
    input_ids = tokenizer.encode(text, return_tensors="pt")

    # Get the embeddings from the token IDs
    embeddings = model.transformer.wte(input_ids)

    # Send the embedding vectors through all transformer layers (for GPT-2, it is 12 layers)
    outputs = model.transformer(inputs_embeds=embeddings)

    # Each layer, applies the self-attention mechanism and goes through a feed-forward NN
    hidden_states = outputs.last_hidden_state

print("Hidden state shape:", hidden_states.shape)

In [ ]:
# Sampling (temperature + top-k)

#    temperature = 0.2 (Set a low temperature value to generate predictable responses)
#    temperature = 1.5 (Set a high temperature value to generate more random and creative responses)
#    top_k = None (full distribution)

def sample_next_token(logits, temperature=1.2, top_k=50):
    logits = logits / temperature

    if top_k is not None:
        values, indices = torch.topk(logits, top_k)
        probs = F.softmax(values, dim=-1)
        choice = torch.multinomial(probs, 1)
        return indices[0, choice]
    else:
        probs = F.softmax(logits, dim=-1)
        return torch.multinomial(probs, 1)

next_token_id = sample_next_token(logits, temperature=0.8, top_k=40)
print("Sampled token:", tokenizer.decode(next_token_id[0]))


In [ ]:
# Logits for the next token. This gives one score per vocabulary token (~50k tokens)

with torch.no_grad():
    last_hidden = hidden_states[:, -1, :]
    logits = model.lm_head(last_hidden)

print("Logits shape:", logits.shape)


In [ ]:
# Full loop (generate multiple tokens)

def generate_step_by_step(prompt, steps=20):
    tokens = tokenizer.encode(prompt, return_tensors="pt")

    for _ in range(steps):
        with torch.no_grad():
            outputs = model(tokens)
            logits = outputs.logits[:, -1, :]
            next_token = sample_next_token(logits, temperature=0.1, top_k=40)

        tokens = torch.cat([tokens, next_token], dim=1)
        print(tokenizer.decode(tokens[0]))

generate_step_by_step(text, steps=20)


Model Coherence: 10

The cat sat on the mat,

The cat sat on the mat, her

The cat sat on the mat, her eyes

The cat sat on the mat, her eyes closed

The cat sat on the mat, her eyes closed,

The cat sat on the mat, her eyes closed, her

The cat sat on the mat, her eyes closed, her head

The cat sat on the mat, her eyes closed, her head resting

The cat sat on the mat, her eyes closed, her head resting on

The cat sat on the mat, her eyes closed, her head resting on the

The cat sat on the mat, her eyes closed, her head resting on the mat

The cat sat on the mat, her eyes closed, her head resting on the mat.

The cat sat on the mat, her eyes closed, her head resting on the mat.

The cat sat on the mat, her eyes closed, her head resting on the mat.


The cat sat on the mat, her eyes closed, her head resting on the mat. "

The cat sat on the mat, her eyes closed, her head resting on the mat. "I

The cat sat on the mat, her eyes closed, her head resting on the mat. "I'm

The cat sat on the mat, her eyes closed, her head resting on the mat. "I'm sorry

The cat sat on the mat, her eyes closed, her head resting on the mat. "I'm sorry,

The cat sat on the mat, her eyes closed, her head resting on the mat. "I'm sorry, I

In [ ]:
# Full loop (generate multiple tokens)

def generate_step_by_step(prompt, steps=20):
    tokens = tokenizer.encode(prompt, return_tensors="pt")

    for _ in range(steps):
        with torch.no_grad():
            outputs = model(tokens)
            logits = outputs.logits[:, -1, :]
            next_token = sample_next_token(logits, temperature=0.8, top_k=40)

        tokens = torch.cat([tokens, next_token], dim=1)
        print(tokenizer.decode(tokens[0]))

generate_step_by_step(text, steps=20)


Model Coherence: 6

The cat sat on the mat,

The cat sat on the mat, but

The cat sat on the mat, but I

The cat sat on the mat, but I just

The cat sat on the mat, but I just couldn

The cat sat on the mat, but I just couldn't

The cat sat on the mat, but I just couldn't get

The cat sat on the mat, but I just couldn't get anything

The cat sat on the mat, but I just couldn't get anything to

The cat sat on the mat, but I just couldn't get anything to do

The cat sat on the mat, but I just couldn't get anything to do.

The cat sat on the mat, but I just couldn't get anything to do. I

The cat sat on the mat, but I just couldn't get anything to do. I couldn

The cat sat on the mat, but I just couldn't get anything to do. I couldn't

The cat sat on the mat, but I just couldn't get anything to do. I couldn't get

The cat sat on the mat, but I just couldn't get anything to do. I couldn't get anything

The cat sat on the mat, but I just couldn't get anything to do. I couldn't get anything to

The cat sat on the mat, but I just couldn't get anything to do. I couldn't get anything to do

The cat sat on the mat, but I just couldn't get anything to do. I couldn't get anything to do.

The cat sat on the mat, but I just couldn't get anything to do. I couldn't get anything to do. I

In [ ]:
# Full loop (generate multiple tokens)

def generate_step_by_step(prompt, steps=20):
    tokens = tokenizer.encode(prompt, return_tensors="pt")

    for _ in range(steps):
        with torch.no_grad():
            outputs = model(tokens)
            logits = outputs.logits[:, -1, :]
            next_token = sample_next_token(logits, temperature=2.0, top_k=40)

        tokens = torch.cat([tokens, next_token], dim=1)
        print(tokenizer.decode(tokens[0]))

generate_step_by_step(text, steps=20)


Model Coherence: 2

The cat sat on the mat like

The cat sat on the mat like something

The cat sat on the mat like something of

The cat sat on the mat like something of an

The cat sat on the mat like something of an infant

The cat sat on the mat like something of an infant as

The cat sat on the mat like something of an infant as its

The cat sat on the mat like something of an infant as its eyes

The cat sat on the mat like something of an infant as its eyes caught

The cat sat on the mat like something of an infant as its eyes caught sight

The cat sat on the mat like something of an infant as its eyes caught sight

The cat sat on the mat like something of an infant as its eyes caught sight

The cat sat on the mat like something of an infant as its eyes caught sight
of

The cat sat on the mat like something of an infant as its eyes caught sight
of its

The cat sat on the mat like something of an infant as its eyes caught sight
of its younger

The cat sat on the mat like something of an infant as its eyes caught sight
of its younger half

The cat sat on the mat like something of an infant as its eyes caught sight
of its younger half and

The cat sat on the mat like something of an infant as its eyes caught sight
of its younger half and the

The cat sat on the mat like something of an infant as its eyes caught sight
of its younger half and the wolf

The cat sat on the mat like something of an infant as its eyes caught sight
of its younger half and the wolf,